# NS07 Tutorial A - Degree Distributions and Heterogeneity

**Lecture:** NS07 - Degree Distributions, Scale-Free Networks, and Robustness

**Reading:** Barabasi, *Network Science*, Chapter 4; Clauset, Shalizi, and Newman (2009)

## Learning goals
- Use the same notation as the lecture slides for discrete degree data: $n_k$, $f_k$, and $p(k)$.
- Compute and interpret the **PMF**, **CDF**, and **CCDF** of a degree distribution.
- Distinguish **heavy-tailed**, **power-law**, and **scale-free** claims.
- Measure degree heterogeneity through
  $$
  \kappa = \frac{\langle k^2 \rangle}{\langle k \rangle}.
  $$
- Understand why **binning** matters for continuous empirical quantities and why CDF/CCDF plots are often more stable.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()


def degree_sequence(G):
    return np.array([degree for _, degree in G.degree()], dtype=int)


def top_airports_by_degree(G, top=10):
    rows = []
    for node, degree in sorted(G.degree, key=lambda item: item[1], reverse=True)[:top]:
        rows.append({
            'airport': node,
            'name': G.nodes[node].get('name', node),
            'degree': degree,
        })
    return pd.DataFrame(rows)


def naive_ccdf_slope(sequence, kmin):
    """A deliberately naive log-log fit used only as a cautionary example."""
    x, y = compute_ccdf(sequence)
    mask = (x >= kmin) & (y > 0)
    if mask.sum() < 3:
        return np.nan
    slope, intercept = np.polyfit(np.log10(x[mask]), np.log10(y[mask]), 1)
    return slope


---
## 1. A real network with visible hubs: US airport routes

We use the **OpenFlights USA** network as the running real example for NS07.
Nodes are airports and edges are direct routes. The degree of an airport is the number of airports it connects to directly.

This is a good NS07 example because:
- degree has a concrete operational meaning,
- hubs are visible immediately,
- heterogeneity has real consequences for traffic concentration and resilience.


In [ ]:
openflights = load_openflights_usa()
geo_pos = positions_from_node_attributes(openflights)
airport_degree = dict(openflights.degree())

plot_metric(
    openflights,
    airport_degree,
    pos=geo_pos,
    with_labels=False,
    colorbar=False,
    min_node_size_px=6,
    max_node_size_px=18,
    width=0.5,
    edge_color='#BBBBBB',
    title='OpenFlights USA: node size and colour encode degree',
)

print_network_stats(openflights)
print(f"kappa = {heterogeneity_kappa(openflights):.2f}")
print('\nTop airports by degree:')
print(top_airports_by_degree(openflights, top=10).to_string(index=False))


---
## 2. Degree distribution as a discrete probability law

For a graph with $n$ nodes:
- $n_k$ = number of nodes with degree $k$,
- $f_k = n_k / n$ = empirical frequency,
- $p(k)$ = degree distribution.

Because degree is **discrete**, this is a probability **mass** function, not a continuous density.


In [ ]:
degrees = degree_sequence(openflights)
support, counts, pmf = compute_pmf(degrees)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.vlines(support, 0, pmf, color=CATEGORY_PALETTE['blue'], alpha=0.75)
ax.scatter(support, pmf, color=CATEGORY_PALETTE['blue'], s=24)
style_axis(
    ax,
    title='OpenFlights USA degree distribution as a PMF',
    xlabel='Degree k',
    ylabel='p(k)',
)
plt.show()

degree_table = pd.DataFrame({
    'k': support,
    'n_k': counts,
    'f_k': pmf,
})
print(degree_table.head(12).round(4).to_string(index=False))


## 3. CDF and CCDF answer different questions

For the same degree data:
- the **CDF** gives $P(K \leq k)$,
- the **CCDF** gives $P(K \geq k)$.

The PMF is useful for exact values, the CDF for threshold questions, and the CCDF for the tail.


In [ ]:
support_cdf, cdf = compute_cdf(degrees)
support_ccdf, ccdf = compute_ccdf(degrees)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].step(support_cdf, cdf, where='post', color=CATEGORY_PALETTE['green'], linewidth=2)
style_axis(
    axes[0],
    title='CDF of airport degree',
    xlabel='Degree threshold k',
    ylabel='P(K <= k)',
)

axes[1].step(support_ccdf, ccdf, where='post', color=CATEGORY_PALETTE['orange'], linewidth=2)
style_axis(
    axes[1],
    title='CCDF of airport degree',
    xlabel='Degree threshold k',
    ylabel='P(K >= k)',
)

plt.show()

p_k_eq_3 = degree_table.loc[degree_table['k'] == 3, 'f_k']
p_k_eq_3 = float(p_k_eq_3.iloc[0]) if len(p_k_eq_3) else 0.0
p_k_le_5 = cdf[np.searchsorted(support_cdf, 5, side='right') - 1]
p_k_ge_20 = ccdf[support_ccdf >= 20][0]

print(f"P(K = 3)    = {p_k_eq_3:.4f}")
print(f"P(K <= 5)   = {p_k_le_5:.4f}")
print(f"P(K >= 20)  = {p_k_ge_20:.4f}")


**Interpretation.**
- Use the **PMF** when the question is exact: "how likely is degree $k$?"
- Use the **CDF** for cumulative thresholds: "what fraction of airports have degree at most $k$?"
- Use the **CCDF** for hubs and tail behaviour: "how rare are airports with degree at least $k$?"


---
## 4. Heterogeneity: compare a real network to two model baselines

The lecture uses degree heterogeneity as the bridge from distributions to consequences.
We compare three graphs with similar size and mean degree:
- **ER** as a narrow random benchmark,
- **BA** as a growth model with hubs,
- **OpenFlights USA** as the real network.


In [ ]:
n_ref = openflights.number_of_nodes()
avg_degree_ref = degrees.mean()
p_ref = avg_degree_ref / (n_ref - 1)
m_ref = max(1, round(avg_degree_ref / 2))

er = nx.erdos_renyi_graph(n_ref, p_ref, seed=RANDOM_SEED)
ba = nx.barabasi_albert_graph(n_ref, m_ref, seed=RANDOM_SEED)

comparison_graphs = {
    'ER': er,
    'BA': ba,
    'OpenFlights USA': openflights,
}

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
for name, graph, color in [
    ('ER', er, CATEGORY_PALETTE['green']),
    ('BA', ba, CATEGORY_PALETTE['orange']),
    ('OpenFlights USA', openflights, CATEGORY_PALETTE['blue']),
]:
    plot_ccdf(
        degree_sequence(graph),
        ax=ax,
        label=name,
        color=color,
        markersize=3,
    )
ax.set_title('Degree CCDF on log-log axes')
ax.legend(frameon=False)
plt.show()

summary_rows = []
for name, graph in comparison_graphs.items():
    degree_vals = degree_sequence(graph)
    summary_rows.append({
        'network': name,
        'n': graph.number_of_nodes(),
        'm': graph.number_of_edges(),
        '<k>': degree_vals.mean(),
        'max k': degree_vals.max(),
        'kappa': heterogeneity_kappa(graph),
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.round(2).to_string(index=False))


## 5. Heavy-tailed is not the same as power-law

The log-log CCDF is a **diagnostic**, not a proof.

Computationally, one quick warning sign is that simple log-log slopes can change a lot when we change the cutoff $k_{\min}$.
If a visual claim is fragile to such choices, we should describe it as **suggestive** or **consistent with**, not as proof.


In [ ]:
slope_rows = []
for name, graph in [('BA', ba), ('OpenFlights USA', openflights)]:
    seq = degree_sequence(graph)
    for kmin in [5, 10, 20, 30]:
        slope_rows.append({
            'network': name,
            'k_min': kmin,
            'naive CCDF slope': naive_ccdf_slope(seq, kmin),
        })

slope_df = pd.DataFrame(slope_rows)
print(slope_df.round(3).to_string(index=False))


**Interpretation.**
- **Heavy-tailed** means large values are not negligible.
- **Power-law** is a specific functional form.
- **Scale-free** is a stronger modelling claim, usually tied to hub-dominated degree distributions.

The slope table above is intentionally naive. Its purpose is pedagogical: a visually broad tail does not automatically justify a precise power-law claim.


---
## 6. Continuous network quantities: betweenness and binning

Betweenness centrality is a **continuous** quantity. For continuous empirical data:
- a histogram estimates a density from bins,
- the chosen bins affect the picture,
- the empirical CDF/CCDF can be built directly from sorted observations.

To keep the tail readable, we focus on airports with **positive** betweenness.


In [ ]:
betweenness = pd.Series(nx.betweenness_centrality(openflights))
positive_betweenness = betweenness[betweenness > 0].to_numpy()
log_bins = np.geomspace(positive_betweenness.min(), positive_betweenness.max(), 20)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(
    positive_betweenness,
    bins=12,
    color=CATEGORY_PALETTE['blue'],
    edgecolor='white',
)
style_axis(
    axes[0, 0],
    title='Betweenness histogram: 12 equal-width bins',
    xlabel='Betweenness',
    ylabel='Count',
)

axes[0, 1].hist(
    positive_betweenness,
    bins=log_bins,
    color=CATEGORY_PALETTE['orange'],
    edgecolor='white',
)
style_axis(
    axes[0, 1],
    title='Betweenness histogram: logarithmic bins',
    xlabel='Betweenness',
    ylabel='Count',
    xscale='log',
)

x_cdf, y_cdf = empirical_continuous_cdf(positive_betweenness)
axes[1, 0].plot(x_cdf, y_cdf, color=CATEGORY_PALETTE['green'], linewidth=2)
style_axis(
    axes[1, 0],
    title='Empirical CDF of positive betweenness',
    xlabel='Betweenness threshold x',
    ylabel='P(B <= x)',
    xscale='log',
)

x_ccdf, y_ccdf = empirical_continuous_ccdf(positive_betweenness)
axes[1, 1].plot(x_ccdf, y_ccdf, color=CATEGORY_PALETTE['red'], linewidth=2)
style_axis(
    axes[1, 1],
    title='Empirical CCDF of positive betweenness',
    xlabel='Betweenness threshold x',
    ylabel='P(B >= x)',
    xscale='log',
    yscale='log',
)

plt.show()

print(f"Positive-betweenness airports: {len(positive_betweenness)} out of {openflights.number_of_nodes()}")
print(f"Median positive betweenness: {np.median(positive_betweenness):.6f}")
print(f"Max positive betweenness   : {positive_betweenness.max():.6f}")


**Interpretation.**
- For continuous quantities, the histogram is useful but **bin-dependent**.
- The CDF and CCDF are usually more stable because they do not require arbitrary bins.
- In a transportation network, broad betweenness values mean that a small number of airports sit on a large fraction of shortest routes.


---
## Takeaway

- Degree in a network is a **discrete** quantity, so the natural object is a PMF $p(k)$.
- The CDF and CCDF answer different threshold questions and are especially useful for tail analysis.
- Broad tails create hubs and raise
  $$
  \kappa = \frac{\langle k^2 \rangle}{\langle k \rangle},
  $$
  which is the lecture summary of heterogeneity.
- A broad log-log tail is not by itself proof of a power law.
- For continuous network quantities such as betweenness, **binning matters**, which is why empirical CDF/CCDF plots are often more robust.
